# TraceCG Benchmark: time-to-target + convergence traces

本 notebook 基于 `benchmark_eigenpro_baselines.ipynb` 的实验框架，新增两部分：

- Part 1: time-to-target（支持 target_delta 列表对比，复用 `accuracy_based_eigenpro_tables.py`）
- Part 2: 自动收敛（PCG/CG 跑到 tol/maxiter；每 10 次迭代记录一次 accuracy，并区分 eigen 阶段与 CG 阶段的 matvec 次数）

数据集：synthetic + USGS LiDAR（Winnebago）。

In [ ]:
# ---- Optional Colab / Drive setup ----
# 本地运行：建议保持 ENABLE_COLAB_DRIVE=False（默认），避免任何联网/挂载行为。
import sys
import time
from pathlib import Path

start_time = time.time()
IS_COLAB = "google.colab" in sys.modules

ENABLE_COLAB_DRIVE = True

DRIVE_MOUNT_DIR = Path("/content/drive")
DRIVE_MYDRIVE_DIR = DRIVE_MOUNT_DIR / "MyDrive"
DRIVE_CACHE_ENABLED = bool(ENABLE_COLAB_DRIVE)
DRIVE_CACHE_SEARCH_RECURSIVE = True
DRIVE_CACHE_CANDIDATE_DIRS = [
    DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "Colab_Experiments" / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR,
]
DRIVE_OUTPUT_DIR = DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_exports"

if IS_COLAB and ENABLE_COLAB_DRIVE:
    try:
        from google.colab import drive
        if not DRIVE_MYDRIVE_DIR.exists():
            drive.mount(str(DRIVE_MOUNT_DIR))
    except Exception as e:
        print("Drive mount skipped:", e)
else:
    if IS_COLAB:
        print("[note] Colab detected, but Drive mount is disabled (ENABLE_COLAB_DRIVE=False)")

print("IS_COLAB:", IS_COLAB)
print("ENABLE_COLAB_DRIVE:", ENABLE_COLAB_DRIVE)
print("DRIVE_MYDRIVE_DIR:", DRIVE_MYDRIVE_DIR)
print("DRIVE_CACHE_ENABLED:", DRIVE_CACHE_ENABLED)


In [ ]:
## For github import

# ---- Optional GitHub import / package install ----
# 本地运行：默认禁用（避免 git clone / pip install 等联网行为）。
# 如需在 Colab 一键安装，把 ENABLE_COLAB_GITHUB_INSTALL=True。
import os
import sys

IS_COLAB = "google.colab" in sys.modules
ENABLE_COLAB_GITHUB_INSTALL = True

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
SUB_DIR = "efgp_eigenpro_py"
PROJECT_PATH = f"/content/{REPO_NAME}"

if IS_COLAB and ENABLE_COLAB_GITHUB_INSTALL:
    if not os.path.exists(PROJECT_PATH):
        !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    else:
        %cd {PROJECT_PATH}
        !git pull origin main

    CODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)
    REPO_ROOT = PROJECT_PATH
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
    if CODE_ROOT not in sys.path:
        sys.path.insert(0, CODE_ROOT)

    print("Installing runtime dependencies")
    !pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com
    !pip install git+https://github.com/EigenPro/EigenPro3.git
    !pip install git+https://github.com/EigenPro/EigenPro-pytorch.git

    requirements_path = os.path.join(CODE_ROOT, "requirements.txt")
    if os.path.exists(requirements_path):
        !pip install -r {requirements_path}

    benchmark_dir_path = os.path.join(CODE_ROOT, "gpu", "benchmark_dataset")
    if os.path.exists(benchmark_dir_path):
        os.chdir(benchmark_dir_path)
        print("cwd:", os.getcwd())
    else:
        print("benchmark_dataset path not found:", benchmark_dir_path)

    os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
    !ldconfig /usr/local/lib

    print("=" * 40)
    try:
        import torch
        import cupy as cp
        import cufinufft
        import eigenpro2
        import eigenpro3
        cp.cuda.Stream.null.synchronize()
        print("PyTorch:", torch.__version__)
        print("GPU:", torch.cuda.get_device_name(0))
        print("cufinufft / eigenpro2 / eigenpro3 import ok")
    except Exception as e:
        print("runtime check failed:", e)
    print("=" * 40)
else:
    print(
        "[note] GitHub/Colab install cell skipped (IS_COLAB=", IS_COLAB,
        ", ENABLE_COLAB_GITHUB_INSTALL=", ENABLE_COLAB_GITHUB_INSTALL, ")"
    )


In [ ]:
import gc
import contextlib
import io
import json
import os
import re
import sys
import time
import traceback
from dataclasses import asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

BENCHMARK_DIR = None
for p in (_here, *_here.parents):
    cand = p / "efgp_eigenpro_py" / "gpu" / "benchmark_dataset"
    if cand.exists():
        BENCHMARK_DIR = cand
        break
if BENCHMARK_DIR is None:
    BENCHMARK_DIR = Path("D:/NU/ML/efgp_eigenpro_py/gpu/benchmark_dataset").resolve()

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.max_columns", 200)

print("cwd:", Path.cwd())
print("sys.path[0]:", sys.path[0])
print("benchmark dir:", BENCHMARK_DIR)

# ---- Local GPU preflight ----
try:
    import cupy as cp

    _HAS_CUPY = True
    try:
        dev = cp.cuda.runtime.getDevice()
        dev_name = cp.cuda.runtime.getDeviceProperties(dev)["name"].decode("utf-8")
    except Exception:
        dev_name = "unknown"
    print("cupy:", cp.__version__)
    print("cupy device:", dev_name)
except Exception as e:
    _HAS_CUPY = False
    print("[warn] cupy import failed; GPU EFGP runs will fail.")
    print("cupy error:", type(e).__name__, str(e))

try:
    import torch

    print("torch:", torch.__version__)
    print("torch cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("torch cuda device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("[note] torch not available:", type(e).__name__, str(e))


## Dataset discovery and selection

沿用 `processed/*.npz`。若在 Colab，可选择从 Drive cache 恢复缺失数据。

In [ ]:
RAW_DATA_DIR = BENCHMARK_DIR
PROCESSED_DATA_DIR = BENCHMARK_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_SUFFIXES = (".npz",)


def discover_processed_datasets(data_dir: Path, suffixes=PROCESSED_DATA_SUFFIXES) -> list[Path]:
    return sorted(
        [p for p in data_dir.iterdir() if p.is_file() and p.suffix.lower() in suffixes],
        key=lambda p: p.name.lower(),
    )


def _short_hash(text: str, n: int = 8) -> str:
    import hashlib

    h = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
    return h[: int(n)]


def _sanitize_dataset_name(name: str) -> str:
    """Sanitize for filenames without dropping suffix after '.'.

    注意：不能用 Path(...).stem，因为字符串里如果含 '.' 会把后半段当作扩展名截断，导致 trace 文件互相覆盖。
    """
    base = str(name)
    base = base.replace("\\", "_").replace("/", "_").replace(":", "_")
    base = re.sub(r"[^0-9a-zA-Z_\-]+", "_", base).strip("_")
    # Avoid too-long filenames on Windows
    if len(base) > 140:
        base = base[:140].rstrip("_")
    return base


DISCOVERED_DATASET_FILES = discover_processed_datasets(PROCESSED_DATA_DIR)
DISCOVERED_DATASET_MAP = {p.stem: p for p in DISCOVERED_DATASET_FILES}
AVAILABLE_DATASET_NAMES = sorted(list(DISCOVERED_DATASET_MAP.keys()))

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("num processed:", len(DISCOVERED_DATASET_FILES))
print("AVAILABLE_DATASET_NAMES (head):", AVAILABLE_DATASET_NAMES[:20])


In [ ]:
# ---- Optional dataset cache restore (Drive) ----
# Default: commented out. Run after DATASET_SELECTION is defined if processed/*.npz lives in Drive.
import shutil

IS_COLAB = "google.colab" in sys.modules
if "DRIVE_CACHE_ENABLED" not in globals():
    DRIVE_MOUNT_DIR = Path("/content/drive")
    DRIVE_MYDRIVE_DIR = DRIVE_MOUNT_DIR / "MyDrive"
    DRIVE_CACHE_ENABLED = bool(IS_COLAB)
    DRIVE_CACHE_SEARCH_RECURSIVE = True
    DRIVE_CACHE_CANDIDATE_DIRS = [
        DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR / "Colab_Experiments" / "EFGP_Eigenpro" / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR,
    ]

if IS_COLAB and DRIVE_CACHE_ENABLED:
    try:
        from google.colab import drive
        if not DRIVE_MYDRIVE_DIR.exists():
            drive.mount(str(DRIVE_MOUNT_DIR))
    except Exception as e:
        print("Drive mount skipped:", e)


def _copy_file_if_missing(src: Path, dst: Path) -> bool:
    if dst.exists() or not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def _find_drive_cached_file(filename: str) -> Path | None:
    if not bool(DRIVE_CACHE_ENABLED):
        return None
    for base in DRIVE_CACHE_CANDIDATE_DIRS:
        if base.exists():
            direct = base / filename
            if direct.exists():
                return direct
    if not bool(DRIVE_CACHE_SEARCH_RECURSIVE) or not DRIVE_MYDRIVE_DIR.exists():
        return None
    try:
        for found in DRIVE_MYDRIVE_DIR.rglob(filename):
            if found.is_file():
                return found
    except Exception as e:
        print(f"drive cache search skipped for {filename}: {e}")
    return None


def restore_dataset_from_drive_cache(dataset_stem: str) -> dict:
    restored = {"npz": False, "json": False, "from": {}}
    for suffix in (".npz", ".json"):
        filename = f"{dataset_stem}{suffix}"
        dst = PROCESSED_DATA_DIR / filename
        src = _find_drive_cached_file(filename)
        if src is None:
            continue
        copied = _copy_file_if_missing(src, dst)
        restored[suffix[1:]] = bool(copied or dst.exists())
        restored["from"][suffix[1:]] = str(src)
        if copied:
            print(f"restored from Drive cache: {dst.name} <- {src}")
    return restored


In [ ]:
# ---- Dataset selection ----
# Synthetic: synthetic_true_func_2d_n{N}
# LiDAR: USGS_LPC_IL_Winnebago_2018_ground_elevation_regression_ntrain{N}

RUN_ALL_DATASETS = False

SYN_N_TRAIN_LIST = [
    #100_000,
    #300_000,
    #1_000_000,
    #3_000_000,
    #10_000_000,
    #30_000_000,
    100_000_000,
]


USGS_BASE_STEM = "USGS_LPC_IL_Winnebago_2018_ground_elevation_regression"
USGS_N_TRAIN_LIST = [
    #100_000,
    #300_000,
    #1_000_000,
    #3_000_000,
    #10_000_000,
    #30_000_000,
    100_000_000,
]

SYNTHETIC_DATASET_STEMS = [f"synthetic_true_func_2d_n{int(n)}" for n in SYN_N_TRAIN_LIST]
USGS_DATASET_STEMS = [f"{USGS_BASE_STEM}_ntrain{int(n)}" for n in USGS_N_TRAIN_LIST]

DATASET_SELECTION_LIST = [
    *SYNTHETIC_DATASET_STEMS,
    *USGS_DATASET_STEMS,
]

if RUN_ALL_DATASETS:
    DATASET_SELECTION = AVAILABLE_DATASET_NAMES[:]
else:
    DATASET_SELECTION = [Path(str(x)).stem for x in DATASET_SELECTION_LIST]

# Optional Drive restore for missing stems
for dataset_stem in sorted(set(DATASET_SELECTION)):
    if dataset_stem not in DISCOVERED_DATASET_MAP:
        restore_dataset_from_drive_cache(dataset_stem)

DISCOVERED_DATASET_FILES = discover_processed_datasets(PROCESSED_DATA_DIR)
DISCOVERED_DATASET_MAP = {p.stem: p for p in DISCOVERED_DATASET_FILES}
AVAILABLE_DATASET_NAMES = sorted(list(DISCOVERED_DATASET_MAP.keys()))

missing = [s for s in DATASET_SELECTION if s not in DISCOVERED_DATASET_MAP]
print("DATASET_SELECTION:", DATASET_SELECTION)
print("missing:", missing)
if missing:
    print("Processed datasets not found under:", PROCESSED_DATA_DIR)
    print("Either copy the matching .npz/.json into processed/, or adjust SYN/USGS N lists.")


## Part 1: time-to-target (target_delta list)

这一部分尽量复用 `accuracy_based_eigenpro_tables.py::run_accuracy_benchmark`，只在外层做 `target_delta` 列表循环与汇总绘图。

In [ ]:
import importlib

import efgp_eigenpro_py.gpu.benchmark_dataset.accuracy_based_eigenpro_tables as acc_tables
acc_tables = importlib.reload(acc_tables)
AccuracyBenchmarkConfig = acc_tables.AccuracyBenchmarkConfig

# ---- Part 1 controls ----
TARGET_DELTA_LIST = [0.02, 0.01,0.001]
TARGET_WINDOW = 3

ACC_EPS_LIST = [1e-5]
ACC_REG_LAMBDA = 0.1
ACC_SOLVE_TOL = 1e-6
ACC_GPU_MAXITER = 100000
ACC_GPU_NUFFT = "auto"
ACC_L2_SCALED = True

ACC_V3_TOPQ_LIST = [45,90,135]
ACC_NYSTROM_TOPQ_LIST = [45,90,135]
ACC_EXTRA_TOPQ_LIST = [45,90,135]

# EigenPro-Nystrom knobs (ours)
ACC_EIGENPRO_NYSTROM_PRECOND_KIND = "coordinate_nystrom"
ACC_EIGENPRO_NYSTROM_REFINE_MODE = "auto"
ACC_EIGENPRO_NYSTROM_SURROGATE_SIZE = 1600
ACC_EIGENPRO_NYSTROM_LOWFREQ_RATIO = 0.5
ACC_EIGENPRO_NYSTROM_OVERSAMPLE = 10
ACC_EIGENPRO_NYSTROM_RITZ_REFINE = True
ACC_EIGENPRO_NYSTROM_SEED = 0
ACC_EIGENPRO_NYSTROM_BLOCK_ROWS = 8192
ACC_EIGENPRO_NYSTROM_RITZ_BLOCK_COLS = 16
ACC_EIGENPRO_NYSTROM_LIFT = True
ACC_EIGENPRO_NYSTROM_REFINE_ITERS = 1
ACC_EIGENPRO_COORD_NYSTROM_GAMMA = 1.0

ACC_EIG_METHOD_TOGGLES = {
    "baseline_v1_topq0": True,
    "baseline_v3_topq": True,
    "nystrom_compact_coordinate": True,
    "extra_rand_range_onepass": True,
}

ACC_MAX_EPOCHS = 8
ACC_N_VAL_EVAL = 20_000
ACC_N_TEST_EVAL = 20_000
ACC_ADD_FASTEST_OURS_ROW = True
ACC_PRINT_EPOCH_PROGRESS = True

# Keep EigenPro baselines off by default in Part1; they live in Part2 trace section.
ACC_EIGENPRO2_ENABLED = False
ACC_EIGENPRO3_ENABLED = False

KERNEL_SPECS = [
    {
        "name": "mat32_ls0.1",
        "family": "matern",
        "nu": 1.5,
        "lengthscale": 0.1,
        "variance": 1.0,
    },
]

DATASET_STEMS_PART1 = [Path(str(s)).stem for s in DATASET_SELECTION]


def apply_part1_cfg_base(cfg: AccuracyBenchmarkConfig) -> AccuracyBenchmarkConfig:
    cfg.dataset_stems = list(DATASET_STEMS_PART1)
    cfg.kernel_specs = list(KERNEL_SPECS)
    cfg.seed_base = 0
    cfg.repeats = 1

    cfg.target_window = int(TARGET_WINDOW)
    cfg.eps_list = list(ACC_EPS_LIST)

    cfg.reg_lambda = float(ACC_REG_LAMBDA)
    cfg.solve_tol = float(ACC_SOLVE_TOL)
    cfg.gpu_maxiter = int(ACC_GPU_MAXITER)
    cfg.gpu_nufft = str(ACC_GPU_NUFFT)
    cfg.l2_scaled = bool(ACC_L2_SCALED)

    cfg.v3_topq_list = list(ACC_V3_TOPQ_LIST)
    cfg.nystrom_topq_list = list(ACC_NYSTROM_TOPQ_LIST)
    cfg.extra_topq_list = list(ACC_EXTRA_TOPQ_LIST)

    cfg.add_fastest_ours_row = bool(ACC_ADD_FASTEST_OURS_ROW)
    cfg.print_epoch_progress = bool(ACC_PRINT_EPOCH_PROGRESS)

    cfg.eig_method_toggles = dict(ACC_EIG_METHOD_TOGGLES)

    cfg.eigenpro_nystrom_precond_kind = str(ACC_EIGENPRO_NYSTROM_PRECOND_KIND)
    cfg.eigenpro_nystrom_refine_mode = str(ACC_EIGENPRO_NYSTROM_REFINE_MODE)
    cfg.eigenpro_nystrom_surrogate_size = int(ACC_EIGENPRO_NYSTROM_SURROGATE_SIZE)
    cfg.eigenpro_nystrom_lowfreq_ratio = float(ACC_EIGENPRO_NYSTROM_LOWFREQ_RATIO)
    cfg.eigenpro_nystrom_oversample = int(ACC_EIGENPRO_NYSTROM_OVERSAMPLE)
    cfg.eigenpro_nystrom_ritz_refine = bool(ACC_EIGENPRO_NYSTROM_RITZ_REFINE)
    cfg.eigenpro_nystrom_seed = int(ACC_EIGENPRO_NYSTROM_SEED)
    cfg.eigenpro_nystrom_block_rows = int(ACC_EIGENPRO_NYSTROM_BLOCK_ROWS)
    cfg.eigenpro_nystrom_ritz_block_cols = int(ACC_EIGENPRO_NYSTROM_RITZ_BLOCK_COLS)
    cfg.eigenpro_nystrom_lift = bool(ACC_EIGENPRO_NYSTROM_LIFT)
    cfg.eigenpro_nystrom_refine_iters = int(ACC_EIGENPRO_NYSTROM_REFINE_ITERS)
    cfg.eigenpro_coord_nystrom_gamma = float(ACC_EIGENPRO_COORD_NYSTROM_GAMMA)

    cfg.max_epochs = int(ACC_MAX_EPOCHS)
    cfg.n_val_eval = int(ACC_N_VAL_EVAL)
    cfg.n_test_eval = int(ACC_N_TEST_EVAL)

    cfg.eigenpro2_enabled = bool(ACC_EIGENPRO2_ENABLED)
    cfg.eigenpro3_enabled = bool(ACC_EIGENPRO3_ENABLED)

    cfg.precompute_methods = {"EFGP-CG": "original", "default": "c1"}
    cfg.precompute_c1_min_n_total = None

    return cfg


print("TARGET_DELTA_LIST:", TARGET_DELTA_LIST)
print("TARGET_WINDOW:", TARGET_WINDOW)
print("datasets:", DATASET_STEMS_PART1)
print("kernels:", [k["name"] for k in KERNEL_SPECS])


In [ ]:
from dataclasses import replace

PART1_RUN = True  # set True to run

part1_all_time_to_target = []
part1_all_same_budget = []
part1_all_history = []
part1_run_meta = []

if PART1_RUN:
    for td in TARGET_DELTA_LIST:
        base = apply_part1_cfg_base(AccuracyBenchmarkConfig())
        cfg_td = replace(base, target_delta=float(td))
        cfg_td.run_tag = datetime.now().strftime(f"tracecg_ttt_td{str(td).replace('.', 'p')}_%Y%m%d_%H%M%S")
        print("=" * 60)
        print("running time-to-target for target_delta=", td, "run_tag=", cfg_td.run_tag)
        res = acc_tables.run_accuracy_benchmark(cfg_td)

        ttt = res["time_to_target_summary"].copy()
        sb = res["same_time_budget_summary"].copy()
        hist = res["raw_eval_history"].copy()
        ttt["target_delta"] = float(td)
        sb["target_delta"] = float(td)
        hist["target_delta"] = float(td)

        part1_all_time_to_target.append(ttt)
        part1_all_same_budget.append(sb)
        part1_all_history.append(hist)
        part1_run_meta.append({"target_delta": float(td), "out_dir": str(res["out_dir"]), "run_tag": cfg_td.run_tag})

    part1_ttt_df = pd.concat(part1_all_time_to_target, ignore_index=True) if part1_all_time_to_target else pd.DataFrame()
    part1_sb_df = pd.concat(part1_all_same_budget, ignore_index=True) if part1_all_same_budget else pd.DataFrame()
    part1_hist_df = pd.concat(part1_all_history, ignore_index=True) if part1_all_history else pd.DataFrame()

    # 重要：只看 head() 很容易“刚好只覆盖第一个 target_delta”，导致误读。
    # 这里先给出每个 delta 的行数与 run_tag 覆盖情况，然后再按 delta 排序展示。
    if not part1_ttt_df.empty and "target_delta" in part1_ttt_df.columns:
        print(
            "part1_ttt_df target_delta counts:\n",
            part1_ttt_df["target_delta"].value_counts(dropna=False).sort_index(),
        )
        if "run_tag" in part1_ttt_df.columns:
            print(
                "part1_ttt_df unique run_tag per delta:\n",
                part1_ttt_df[["target_delta", "run_tag"]]
                .drop_duplicates()
                .sort_values(["target_delta", "run_tag"], kind="mergesort"),
            )

    # 完整展示（通过调大 pandas 显示阈值；表太大时 Colab 仍可能折叠，但内容已完整在 DataFrame 内）
    with pd.option_context("display.max_rows", 200, "display.max_columns", 120, "display.width", None):
        if not part1_ttt_df.empty and "target_delta" in part1_ttt_df.columns:
            display(
                part1_ttt_df.sort_values(
                    ["target_delta", "dataset", "kernel", "method"], kind="mergesort"
                )
            )
        else:
            display(part1_ttt_df)

        if not part1_sb_df.empty and "target_delta" in part1_sb_df.columns:
            display(
                part1_sb_df.sort_values(
                    ["target_delta", "dataset", "kernel", "method"], kind="mergesort"
                )
            )
        else:
            display(part1_sb_df)

        display(part1_hist_df.tail(200))

    # ---- 保存 Part1 合并后的三张表 + 元信息（用于打包 zip 并下载） ----
    from pathlib import Path
    import json

    part1_merged_dir = (BENCHMARK_DIR / "outputs" / datetime.now().strftime("tracecg_part1_merged_%Y%m%d_%H%M%S")).resolve()
    part1_merged_dir.mkdir(parents=True, exist_ok=True)

    part1_ttt_path = part1_merged_dir / "part1_time_to_target_merged.csv"
    part1_sb_path = part1_merged_dir / "part1_same_time_budget_merged.csv"
    part1_hist_path = part1_merged_dir / "part1_raw_eval_history_merged.csv"
    part1_meta_path = part1_merged_dir / "part1_run_meta.json"

    part1_ttt_df.to_csv(part1_ttt_path, index=False)
    part1_sb_df.to_csv(part1_sb_path, index=False)
    part1_hist_df.to_csv(part1_hist_path, index=False)

    part1_export_meta = {
        "created_at": datetime.now().isoformat(),
        "merged_dir": str(part1_merged_dir),
        "tables": {
            "time_to_target": str(part1_ttt_path),
            "same_time_budget": str(part1_sb_path),
            "raw_eval_history": str(part1_hist_path),
        },
        "runs": list(part1_run_meta),
    }
    part1_meta_path.write_text(json.dumps(part1_export_meta, indent=2), encoding="utf-8")

    # 供 export cell 统一打包
    part1_merged_out_dir = str(part1_merged_dir)
    print("[part1] merged tables saved to:", part1_merged_dir)
else:
    print("PART1_RUN=False. Set True to run time-to-target.")


In [ ]:
# ---- Part 1 plots ----
# 画“真实指标 vs 时间”：x=best_val_rmse_std（或 val_rmse_std），y=time_to_target。
# target_delta 只作为对比维度（颜色/分组），不作为坐标轴指标。
if "part1_ttt_df" in globals() and isinstance(part1_ttt_df, pd.DataFrame) and not part1_ttt_df.empty:
    df = part1_ttt_df.copy()

    ykey = "wall_time_to_target_s" if "wall_time_to_target_s" in df.columns else "fit_time_to_target_s"
    xkey = "best_val_rmse_std" if "best_val_rmse_std" in df.columns else "val_rmse_std"

    # 只画成功且（如果有的话）达标的点
    if "status" in df.columns:
        df = df[df["status"].astype(str) == "ok"].copy()
    if "reached_target" in df.columns:
        df = df[df["reached_target"].astype(bool)].copy()

    if df.empty or (xkey not in df.columns) or (ykey not in df.columns):
        print("Part1 plot skipped (missing columns or empty after filter).")
    else:
        for (ds, ker), sub0 in df.groupby(["dataset", "kernel"], dropna=False):
            fig, ax = plt.subplots(1, 1, figsize=(10, 4.5), constrained_layout=True)
            for (meth, td), sub in sub0.groupby(["method", "target_delta"], dropna=False):
                xs = sub[xkey].astype(float).values
                ys = sub[ykey].astype(float).values
                ok = np.isfinite(xs) & np.isfinite(ys)
                if not ok.any():
                    continue
                ax.scatter(xs[ok], ys[ok], s=40, alpha=0.8, label=f"{meth} | delta={td}")
            ax.set_xlabel(xkey)
            ax.set_ylabel(ykey)
            ax.set_title(f"Part1 time-to-target: {ds} | {ker}")
            ax.grid(True, alpha=0.25)
            ax.legend()
            plt.show()
else:
    print("Part1 plot skipped (no part1_ttt_df).")


## Part 2: automatic convergence + TraceCG (per-10-iter accuracy)

这一部分实现：

- PCG/CG 跑到 tol 或 maxiter
- 每 10 次迭代在一个较小的 eval 子集上计算 accuracy
- 记录 matvec 次数：
  - eigen_matvec_calls: eigenspace/预条件子构建阶段用到的 `A*v` 次数
  - cg_matvec_calls: PCG 求解阶段的 `A*v` 次数

In [ ]:
# ---- Part 2 imports (reuse existing code) ----
from efgp_eigenpro_py.kernels import make_matern, make_squared_exponential
from efgp_eigenpro_py.efgp_solver import EFGPSolver

from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext, ensure_gpu_data_context
from efgp_eigenpro_py.gpu.v1_ops import gpu_precompute_v1, apply_A_v1, predict_v1
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig, estimate_top_eigenspace_v3, mu_for_precond_from_eig
from efgp_eigenpro_py.gpu.v2_preconditioner import (
    GPUPreconditionerData,
    EnsembleCoordinateNystromPreconditionerData,
    apply_preconditioner_v2,
    apply_preconditioner_coordinate_nystrom,
    apply_preconditioner_ensemble_coordinate_nystrom,
    apply_preconditioner_hybrid_topr_coordinate,
    build_coordinate_nystrom_preconditioner_data,
    build_ensemble_coordinate_nystrom_preconditioner_data,
    HybridToprCoordinatePreconditionerData,
)

# reuse dataset helpers + metrics from acc_tables
load_dataset = acc_tables.load_dataset
split_train_val = acc_tables.split_train_val
regression_metrics_std = acc_tables.regression_metrics_std

try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

print("torch available:", _HAS_TORCH)


In [ ]:
def make_efgp_kernel(kernel_cfg: dict, dim: int):
    fam = str(kernel_cfg.get("family", "")).strip().lower()
    lengthscale = float(kernel_cfg["lengthscale"])
    variance = float(kernel_cfg.get("variance", 1.0))
    if fam in ("matern", "mat"):
        return make_matern(
            lengthscale=lengthscale,
            nu=float(kernel_cfg.get("nu", 1.5)),
            dim=int(dim),
            variance=variance,
        )
    if fam in ("gaussian", "se", "squared_exponential", "squared-exponential", "rbf"):
        return make_squared_exponential(lengthscale=lengthscale, dim=int(dim), variance=variance)
    raise ValueError(f"unsupported kernel family: {fam!r}")


def _clear_state():
    gc.collect()
    if _HAS_TORCH and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _as_float64_np(x: np.ndarray) -> np.ndarray:
    return np.asarray(x, dtype=np.float64)


def _as_1d_float64_np(x: np.ndarray) -> np.ndarray:
    return np.asarray(x, dtype=np.float64).reshape(-1)


In [ ]:
def pcg_solve_gpu_traced(
    backend,
    matvec,
    precond,
    b,
    op_ctx,
    tol: float,
    maxiter: int,
    *,
    trace_every: int = 10,
    trace_eval_fn=None,
    return_stats: bool = True,
    work_prefix: str = "pcg_trace",
):
    """Minimal copy of pcg_solve_gpu with trace hook.

    - trace_eval_fn(x_gpu, it, relres) -> dict metrics
    - counts matvec/precond calls
    """
    xp = backend.xp
    dtype = xp.complex128
    b = xp.asarray(b, dtype=dtype).reshape(-1)
    n = int(b.size)

    def _buf(name: str):
        buf = getattr(op_ctx, name, None)
        if buf is None or int(getattr(buf, "size", 0)) != int(n) or buf.dtype != dtype:
            buf = xp.empty((int(n),), dtype=dtype)
            setattr(op_ctx, name, buf)
        return buf

    x = _buf(f"{work_prefix}_x")
    r = _buf(f"{work_prefix}_r")
    p = _buf(f"{work_prefix}_p")
    Ap = _buf(f"{work_prefix}_ap")
    z = _buf(f"{work_prefix}_z")

    t_matvec_total = 0.0
    t_precond_total = 0.0
    n_matvec = 0
    n_precond = 0

    def _sync():
        cuda = getattr(xp, "cuda", None)
        if cuda is not None:
            cuda.Stream.null.synchronize()

    def _matvec_in(v, out):
        nonlocal t_matvec_total, n_matvec
        _sync()
        t0 = time.perf_counter()
        matvec(v, out)
        _sync()
        t_matvec_total += time.perf_counter() - t0
        n_matvec += 1

    def _precond_in(v, out):
        nonlocal t_precond_total, n_precond
        _sync()
        t0 = time.perf_counter()
        precond(v, out)
        _sync()
        t_precond_total += time.perf_counter() - t0
        n_precond += 1

    trace_rows = []

    x.fill(0)
    _matvec_in(x, Ap)
    xp.subtract(b, Ap, out=r)
    _precond_in(r, z)
    xp.copyto(p, z)

    rzold = float(xp.real(backend.linalg.vdot(r, z)))
    norm_b = max(float(backend.linalg.norm(b)), 1e-30)

    def _maybe_trace(it: int):
        if trace_eval_fn is None:
            return
        if it == 0 or (trace_every > 0 and (it % int(trace_every) == 0)):
            rel = float(backend.linalg.norm(r) / norm_b)
            extra = trace_eval_fn(x, int(it), float(rel))
            row = {"iter": int(it), "relres": float(rel)}
            if isinstance(extra, dict):
                row.update(extra)
            trace_rows.append(row)

    _maybe_trace(0)

    it = 0
    for k in range(1, int(maxiter) + 1):
        it = k
        _matvec_in(p, Ap)
        denom = float(xp.real(backend.linalg.vdot(p, Ap)))
        if denom <= 0.0 or not np.isfinite(denom):
            raise RuntimeError(f"PCG denom non-positive or non-finite: {denom}")
        alpha = rzold / denom
        x += alpha * p
        r -= alpha * Ap
        rel = float(backend.linalg.norm(r) / norm_b)
        _maybe_trace(it)
        if rel < float(tol):
            break
        _precond_in(r, z)
        rznew = float(xp.real(backend.linalg.vdot(r, z)))
        beta = rznew / max(rzold, 1e-30)
        xp.multiply(p, beta, out=Ap)
        Ap += z
        xp.copyto(p, Ap)
        rzold = rznew

    relres = float(backend.linalg.norm(r) / norm_b)
    stats = {
        "n_matvec": int(n_matvec),
        "t_matvec_total": float(t_matvec_total),
        "t_matvec_avg": float(t_matvec_total / max(n_matvec, 1)),
        "n_precond": int(n_precond),
        "t_precond_total": float(t_precond_total),
        "t_precond_avg": float(t_precond_total / max(n_precond, 1)),
    }
    return x, int(it), float(relres), stats, pd.DataFrame(trace_rows)


In [ ]:
def run_traced_v3_case(
    *,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val_eval: np.ndarray,
    y_val_eval: np.ndarray,
    y_std: float,
    kernel_cfg: dict,
    eps: float,
    reg_lambda: float,
    solve_tol: float,
    solve_maxiter: int,
    eig_cfg: EigenspaceConfig | None,
    trace_every: int,
    tag: str,
    precompute_method_requested: str,
):
    """Run one traced EFGP solve (v3 style), returning summary + trace + counters."""

    x_train64 = _as_float64_np(x_train)
    y_train64 = _as_1d_float64_np(y_train)

    kernel = make_efgp_kernel(kernel_cfg, int(x_train64.shape[1]))
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=float(reg_lambda),
        eps=float(eps),
        nufft_tol=1e-10,
        l2scaled=True,
    )

    backend = build_gpu_backend_bundle(BackendConfig(nufft="auto"))
    data_ctx = ensure_gpu_data_context(backend, x_train64, y_train64, state=None)
    op_ctx = GPUOperatorContext()

    # Precompute policy (与你的要求一致)：v1/original，其他强制 c1。
    # 复用 Part1 的 binned precompute patch：通过 acc_tables._BENCHMARK_PC_METHOD_ACTIVE 控制。
    global _PART2_PC_PATCHED
    if "_PART2_PC_PATCHED" not in globals():
        _PART2_PC_PATCHED = False
    if not bool(_PART2_PC_PATCHED):
        try:
            acc_tables.install_gpu_precompute_patch(AccuracyBenchmarkConfig())
        except Exception as _e:
            print("[warn] install_gpu_precompute_patch failed; falling back to original precompute.", _e)
        _PART2_PC_PATCHED = True

    pcm_req = str(precompute_method_requested).strip().lower()
    if pcm_req not in ("original", "c1", "c0", "c2"):
        pcm_req = "original"

    prev_pcm = getattr(acc_tables, "_BENCHMARK_PC_METHOD_ACTIVE", None)
    t0 = time.perf_counter()
    try:
        acc_tables._BENCHMARK_PC_METHOD_ACTIVE = pcm_req
        data_ctx = gpu_precompute_v1(
            backend,
            solver.kernel,
            solver.eps,
            solver.nufft_tol,
            data_ctx,
            op_ctx,
            l2scaled=solver.l2scaled,
            chunk_size=None,
        )
        pc_patch_extra = dict(getattr(acc_tables, "_LAST_PC_PATCH_EXTRA", {}) or {})
    finally:
        acc_tables._BENCHMARK_PC_METHOD_ACTIVE = prev_pcm
        try:
            acc_tables._LAST_PC_PATCH_EXTRA.clear()
        except Exception:
            pass
    t1 = time.perf_counter()

    # --- eigen stage (optional) ---
    eigen_matvec_calls = 0

    def apply_A_block_counted(v_block):
        nonlocal eigen_matvec_calls
        xp = backend.xp
        vb = xp.asarray(v_block, dtype=xp.complex128)
        if vb.ndim == 1:
            vb = vb.reshape(-1, 1)
        out_block = xp.empty_like(vb)
        for i in range(vb.shape[1]):
            apply_A_v1(backend, data_ctx, vb[:, i], float(reg_lambda), op_ctx, out=out_block[:, i])
            eigen_matvec_calls += 1
        return out_block

    precond_data = None
    precond_kind = "none"
    t2 = t1
    t3 = t1

    if eig_cfg is not None and int(eig_cfg.q_max) > 0:
        # align eigenpro_nystrom requirements
        method_name = str((eig_cfg.eig_method if eig_cfg.eig_method is not None else eig_cfg.method) or "subspace_iter").lower()
        if method_name in (
            "eigenpro_nystrom",
            "nystrom",
            "ep_nystrom",
            "coordinate_nystrom",
            "coord_nystrom",
            "ensemble_coordinate_nystrom",
            "random_support_lift",
            "support_lift",
        ):
            eig_cfg.method_cfg = dict(eig_cfg.method_cfg or {})
            eig_cfg.method_cfg.setdefault("data_ctx", data_ctx)
            eig_cfg.method_cfg.setdefault("reg_lambda", float(reg_lambda))

        vals_gpu, vecs_gpu, eig_diag = estimate_top_eigenspace_v3(
            backend=backend,
            apply_A_block_gpu=apply_A_block_counted,
            size=int(data_ctx.rhs_gpu.size),
            cfg=eig_cfg,
        )
        t2 = time.perf_counter()

        q = int(eig_cfg.q_max)
        precond_kind = str(eig_diag.get("precond_kind", "full_eigenpro")).lower()
        if precond_kind in ("coordinate_nystrom", "diag_coordinate_nystrom"):
            coord_gamma = float(eig_diag.get("coord_nystrom_gamma", 1.0))
            precond_data = build_coordinate_nystrom_preconditioner_data(
                backend,
                eig_diag["S_gpu"],
                eig_diag["V_gpu"],
                eig_diag["theta_gpu"],
                float(eig_diag["mu"]),
                gamma=coord_gamma,
                diag_inv_sqrt_gpu=eig_diag.get("diag_inv_sqrt_gpu", None),
            )
        elif precond_kind == "ensemble_coordinate_nystrom":
            precond_data = build_ensemble_coordinate_nystrom_preconditioner_data(
                backend,
                list(eig_diag.get("ensemble_entries", []) or []),
                gamma=float(eig_diag.get("ensemble_gamma", 1.0)),
            )
        elif precond_kind in ("hybrid_topr_coordinate", "hybrid_top_r_coordinate", "topr_hybrid_coordinate"):
            xp = backend.xp
            coord_gamma = float(eig_diag.get("coord_nystrom_gamma", 1.0))
            mu_dense = float(eig_diag.get("mu"))
            evals_r = xp.asarray(eig_diag.get("hybrid_dense_eigvals_gpu"))
            U_r = xp.asarray(eig_diag.get("hybrid_dense_eigvecs_gpu"))
            scale_dense = xp.ascontiguousarray(1.0 - (mu_dense / xp.asarray(evals_r)))
            dense = GPUPreconditionerData(
                U_gpu=xp.ascontiguousarray(U_r),
                UH_gpu=xp.ascontiguousarray(U_r.conj().T),
                scale_gpu=scale_dense,
                scale_col_gpu=scale_dense.reshape(-1, 1),
            )
            tail = build_coordinate_nystrom_preconditioner_data(
                backend,
                eig_diag["hybrid_tail_S_gpu"],
                eig_diag["hybrid_tail_V_gpu"],
                eig_diag["hybrid_tail_theta_gpu"],
                float(eig_diag.get("hybrid_tail_mu", eig_diag.get("mu"))),
                gamma=coord_gamma,
                diag_inv_sqrt_gpu=eig_diag.get("diag_inv_sqrt_gpu", None),
            )
            precond_data = HybridToprCoordinatePreconditionerData(dense=dense, tail=tail)
        else:
            mu = mu_for_precond_from_eig(vals_gpu, q, eig_diag)
            scale_gpu = backend.xp.asarray(1.0 - (mu / vals_gpu[:q]))
            precond_data = GPUPreconditionerData(
                U_gpu=vecs_gpu[:, :q],
                UH_gpu=vecs_gpu[:, :q].conj().T,
                scale_gpu=scale_gpu,
                scale_col_gpu=scale_gpu.reshape(-1, 1),
            )
        t3 = time.perf_counter()

    # --- PCG stage ---
    cg_matvec_calls = 0

    def _matvec(v, out):
        nonlocal cg_matvec_calls
        apply_A_v1(backend, data_ctx, v, float(reg_lambda), op_ctx, out=out)
        cg_matvec_calls += 1

    def _precond(v, out):
        if precond_data is None:
            # identity for unprecond runs
            backend.xp.copyto(out, backend.xp.asarray(v))
            return
        if hasattr(precond_data, "tail"):
            apply_preconditioner_hybrid_topr_coordinate(backend, precond_data, v, op_ctx=op_ctx, out=out)
        elif precond_kind == "ensemble_coordinate_nystrom" or isinstance(precond_data, EnsembleCoordinateNystromPreconditionerData):
            apply_preconditioner_ensemble_coordinate_nystrom(backend, precond_data, v, op_ctx=op_ctx, out=out)
        elif precond_kind in ("coordinate_nystrom", "diag_coordinate_nystrom") or all(
            hasattr(precond_data, k) for k in ("S_gpu", "V_gpu", "VH_gpu")
        ):
            apply_preconditioner_coordinate_nystrom(backend, precond_data, v, op_ctx=op_ctx, out=out)
        else:
            apply_preconditioner_v2(backend, precond_data, v, op_ctx=op_ctx, out=out)

    # trace eval uses current beta to predict on a small eval subset
    x_val_eval64 = _as_float64_np(x_val_eval)
    y_val_eval64 = np.asarray(y_val_eval, dtype=np.float64).reshape(-1, 1)

    def trace_eval_fn(beta_gpu, it: int, relres: float):
        yhat = predict_v1(backend, data_ctx, x_val_eval64, beta_gpu)
        xp = backend.xp
        if hasattr(xp, "asnumpy"):
            yhat_np = np.asarray(xp.asnumpy(yhat)).reshape(-1, 1)
        else:
            yhat_np = np.asarray(yhat).reshape(-1, 1)
        m = regression_metrics_std(y_val_eval64, yhat_np, float(y_std))
        return {
            "val_rmse_std": float(m["rmse_std"]),
            "val_mae_std": float(m["mae_std"]),
            "val_r2": float(m["r2"]),
        }

    t4 = time.perf_counter()
    beta_gpu, it, relres, stats, trace_df = pcg_solve_gpu_traced(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        tol=float(solve_tol),
        maxiter=int(solve_maxiter),
        trace_every=int(trace_every),
        trace_eval_fn=trace_eval_fn,
    )
    t5 = time.perf_counter()

    summary = {
        "tag": str(tag),
        "status": "ok",
        "precond_kind": str(precond_kind),
        "eig_q": int(eig_cfg.q_max) if eig_cfg is not None else 0,
        "solve_tol": float(solve_tol),
        "solve_maxiter": int(solve_maxiter),
        "cg_iters": int(it),
        "cg_relres": float(relres),
        "time_precompute": float(t1 - t0),
        "time_eigenspace": float(t2 - t1),
        "time_precond_build": float(t3 - t2),
        "time_solve": float(t5 - t4),
        "time_total": float(t5 - t0),
        "eigen_matvec_calls": int(eigen_matvec_calls),
        "cg_matvec_calls": int(cg_matvec_calls),
        "pcg_n_matvec": int(stats.get("n_matvec", -1)),
        "pcg_n_precond": int(stats.get("n_precond", -1)),
        # 记录 precompute 口径（满足“v1 original / others c1”可审计）
        "precompute_method_requested": str(pcm_req),
        "precompute_method_effective": str(pc_patch_extra.get("precompute_method_effective", pcm_req)),
        "nufft_stage": str(getattr(data_ctx, "meta", {}).get("nufft_stage", "")),
        "pc_time_precompute_NUFFT": float(pc_patch_extra.get("time_precompute_NUFFT", float("nan"))),
        "pc_time_precompute_binned": float(pc_patch_extra.get("time_precompute_binned", float("nan"))),
    }

    return summary, trace_df


In [ ]:
# ---- Part 2 run config ----
PART2_RUN = True  # set True to run

TRACE_EVERY = 10
TRACE_VAL_EVAL = 5000  # smaller eval for per-iter tracing

PART2_EPS = 1e-5
PART2_REG_LAMBDA = 0.1
PART2_SOLVE_TOL = 1e-6
PART2_SOLVE_MAXITER = 4000

TOPQ_LIST = [45,90,135]
NYSTROM_SURROGATE_SIZE = 1600

# Methods to compare in Part2 (EFGP PCG)
# - unprecond: eig_cfg=None => identity precond
# - full_eigenpro: subspace_iter (经典 block power / subspace iter)
# - coord_nystrom: eigenpro_nystrom with compact coordinate precond
# - rand_range_onepass: randomized range (one-pass) eigenspace estimate

def make_eig_cfg_full(top_q: int):
    return EigenspaceConfig(q_max=int(top_q), block_size=int(top_q + 16), n_iter=3, method="subspace_iter")


def make_eig_cfg_coord_nystrom(top_q: int):
    return EigenspaceConfig(
        q_max=int(top_q),
        block_size=int(top_q + 16),
        n_iter=3,
        eig_method="eigenpro_nystrom",
        method_cfg={
            "precond_kind": "coordinate_nystrom",
            "coord_nystrom_gamma": 1.0,
        },
        surrogate_size=int(NYSTROM_SURROGATE_SIZE),
        surrogate_lowfreq_ratio=0.5,
        surrogate_oversample=10,
        surrogate_seed=0,
        surrogate_block_rows=8192,
        surrogate_ritz_refine=True,
        surrogate_ritz_block_cols=16,
        surrogate_lift=True,
        surrogate_refine_mode="auto",
        surrogate_refine_iters=1,
    )


def make_eig_cfg_rand_range_onepass(top_q: int):
    # 对应 v3_eigenspace.py::_estimate_via_extra_rand_range_onepass
    # 这里默认用 full_eigenpro 预条件子（precond_kind 默认就是 full_eigenpro）。
    return EigenspaceConfig(
        q_max=int(top_q),
        block_size=int(top_q + 16),
        n_iter=3,
        eig_method="rand_range_onepass",
        method_cfg={
            "oversample": 16,
            "power_iters": 0,
            "omega_kind": "gaussian",
            "block_cols": 16,
            "final_ritz": True,
            "sparsity": 8,
            "seed": 0,
        },
    )


PART2_METHODS = [
    # unpreconditioned CG 只有一个配置：q=0
    {"name": "unprecond_cg", "eig_cfg_fn": lambda q: None, "top_q": 0},
]

# 其余方法：对 TOPQ_LIST 全部展开
for _q in TOPQ_LIST:
    q = int(_q)
    PART2_METHODS.extend(
        [
            {"name": "full_eigenpro", "eig_cfg_fn": make_eig_cfg_full, "top_q": q},
            {"name": "coord_nystrom", "eig_cfg_fn": make_eig_cfg_coord_nystrom, "top_q": q},
            {"name": "rand_range_onepass", "eig_cfg_fn": make_eig_cfg_rand_range_onepass, "top_q": q},
        ]
    )

print("PART2_METHODS (name, top_q):", [(m["name"], int(m.get("top_q", 0))) for m in PART2_METHODS])


In [ ]:
def part2_load_payloads():
    payloads = []
    for stem in DATASET_SELECTION:
        stem = Path(str(stem)).stem
        if stem not in DISCOVERED_DATASET_MAP:
            continue
        payloads.append(acc_tables.load_dataset(stem))
    return payloads


def part2_make_split(payload: dict):
    cfg = AccuracyBenchmarkConfig()
    cfg.train_core_fraction = 0.9
    cfg.n_val_eval = int(TRACE_VAL_EVAL)
    cfg.n_test_eval = int(TRACE_VAL_EVAL)
    cfg.seed_base = 0
    return split_train_val(payload, cfg, seed=0)


def part2_run_all():
    payloads = part2_load_payloads()
    if not payloads:
        raise RuntimeError("No datasets available for Part2. Check processed/.npz.")

    out_tag = datetime.now().strftime("tracecg_part2_%Y%m%d_%H%M%S")
    out_dir = BENCHMARK_DIR / "outputs" / out_tag
    out_dir.mkdir(parents=True, exist_ok=True)
    trace_dir = out_dir / "traces"
    trace_dir.mkdir(parents=True, exist_ok=True)

    all_summary = []
    all_traces = []

    for payload in payloads:
        split = part2_make_split(payload)
        x_train = split["x_train_core"]
        y_train = split["y_train_core"].reshape(-1)
        x_val = split["x_val_eval"]
        y_val = split["y_val_eval"]

        for kernel_cfg in KERNEL_SPECS:
            for m in PART2_METHODS:
                name = str(m["name"])
                q = int(m.get("top_q", 0))
                eig_cfg = m["eig_cfg_fn"](q) if callable(m.get("eig_cfg_fn")) else None
                tag = f"{payload['name']}_{kernel_cfg['name']}_{name}_q{q}"
                print("-" * 80)
                print("Part2 run:", tag)

                try:
                    pcm_req = "original" if name == "unprecond_cg" else "c1"
                    summary, trace_df = run_traced_v3_case(
                        x_train=x_train,
                        y_train=y_train,
                        x_val_eval=x_val,
                        y_val_eval=y_val,
                        y_std=float(payload.get("y_std", np.nan)),
                        kernel_cfg=kernel_cfg,
                        eps=float(PART2_EPS),
                        reg_lambda=float(PART2_REG_LAMBDA),
                        solve_tol=float(PART2_SOLVE_TOL),
                        solve_maxiter=int(PART2_SOLVE_MAXITER),
                        eig_cfg=eig_cfg,
                        trace_every=int(TRACE_EVERY),
                        tag=tag,
                        precompute_method_requested=pcm_req,
                    )
                    summary_row = {
                        "dataset": payload["name"],
                        "kernel": kernel_cfg["name"],
                        "method": name,
                        "top_q": q,
                        **summary,
                    }
                    all_summary.append(summary_row)

                    trace_df = trace_df.copy()
                    trace_df["dataset"] = payload["name"]
                    trace_df["kernel"] = kernel_cfg["name"]
                    trace_df["method"] = name
                    trace_df["top_q"] = q
                    trace_df["tag"] = tag
                    all_traces.append(trace_df)

                    # Use sanitized name + hash to avoid collisions (and Windows path quirks)
                    trace_fname = f"{_sanitize_dataset_name(tag)}_{_short_hash(tag)}_trace.csv"
                    trace_path = trace_dir / trace_fname
                    trace_df.to_csv(trace_path, index=False)
                    print("trace saved:", trace_path)

                except Exception as exc:
                    traceback.print_exc()
                    all_summary.append(
                        {
                            "dataset": payload["name"],
                            "kernel": kernel_cfg["name"],
                            "method": name,
                            "top_q": q,
                            "tag": tag,
                            "status": "error",
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )
                finally:
                    _clear_state()

    summary_df = pd.DataFrame(all_summary)
    traces_df = pd.concat(all_traces, ignore_index=True) if all_traces else pd.DataFrame()

    summary_path = out_dir / "summary.csv"
    traces_path = out_dir / "trace_all.csv"
    cfg_path = out_dir / "config.json"

    summary_df.to_csv(summary_path, index=False)
    if not traces_df.empty:
        traces_df.to_csv(traces_path, index=False)
    methods_serializable = []
    for m in PART2_METHODS:
        ms = dict(m)
        # drop callables for JSON
        if "eig_cfg_fn" in ms:
            ms.pop("eig_cfg_fn", None)
        methods_serializable.append(ms)

    cfg_payload = {
        "timestamp": datetime.now().isoformat(),
        "out_tag": out_tag,
        "datasets": [p["name"] for p in payloads],
        "kernels": KERNEL_SPECS,
        "methods": methods_serializable,
        "trace_every": int(TRACE_EVERY),
        "trace_val_eval": int(TRACE_VAL_EVAL),
        "eps": float(PART2_EPS),
        "reg_lambda": float(PART2_REG_LAMBDA),
        "solve_tol": float(PART2_SOLVE_TOL),
        "solve_maxiter": int(PART2_SOLVE_MAXITER),
    }
    cfg_path.write_text(json.dumps(cfg_payload, indent=2), encoding="utf-8")

    print("summary:", summary_path)
    print("traces:", traces_path)
    print("config:", cfg_path)
    return out_dir, summary_df, traces_df


if PART2_RUN:
    part2_out_dir, part2_summary_df, part2_traces_df = part2_run_all()

    # 展示可以截断，但落盘/打包的是完整 CSV。
    print("[part2] summary rows:", int(getattr(part2_summary_df, "shape", (0, 0))[0]))
    print("[part2] trace rows:", int(getattr(part2_traces_df, "shape", (0, 0))[0]))
    print("[part2] out_dir:", part2_out_dir)

    with pd.option_context("display.max_rows", 30, "display.max_columns", 120, "display.width", None):
        display(part2_summary_df)
        display(part2_traces_df.head(20))
else:
    print("PART2_RUN=False. Set True to run Part2 trace experiments.")


In [ ]:
# ---- Part 2 plots ----
# 诉求：
# - 纵轴 RMSE 用对数坐标，便于比较后期微小差异
# - 横轴迭代数更关注前 200 次：用 symlog，并把线性阈值设为 200（0~200 线性，之后对数）
# - 产物：除 trace.csv 外，也要把图保存到输出目录，便于复现/打包
from pathlib import Path
from datetime import datetime

if "part2_traces_df" in globals() and isinstance(part2_traces_df, pd.DataFrame) and not part2_traces_df.empty:
    df = part2_traces_df.copy()

    if "part2_out_dir" in globals() and part2_out_dir is not None:
        _fig_root = Path(part2_out_dir).resolve()
    else:
        _fig_root = (Path(BENCHMARK_DIR) / "outputs" / datetime.now().strftime("tracecg_part2_figures_%Y%m%d_%H%M%S")).resolve()
    fig_dir = _fig_root / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    print("[part2] figures dir:", fig_dir)

    for (ds, ker), sub0 in df.groupby(["dataset", "kernel"], dropna=False):
        fig, ax = plt.subplots(1, 1, figsize=(10, 4.5), constrained_layout=True)

        for (meth, q), sub in sub0.groupby(["method", "top_q"], dropna=False):
            if "val_rmse_std" not in sub.columns:
                continue

            x = np.asarray(sub["iter"], dtype=float)
            y = np.asarray(sub["val_rmse_std"], dtype=float)
            ok = np.isfinite(x) & np.isfinite(y) & (y > 0)
            if not ok.any():
                continue

            ax.plot(x[ok], y[ok], marker="o", markersize=4, linewidth=2, label=f"{meth} q={int(q)}")

        ax.set_xlabel("cg_iter")
        ax.set_ylabel("val_rmse_std")
        ax.set_title(f"Part2 trace: {ds} | {ker}")

        # x: emphasize early iterations
        ax.set_xscale("symlog", linthresh=200, linscale=1.0, base=10)
        # y: log scale
        ax.set_yscale("log")

        ax.grid(True, which="both", alpha=0.25)
        ax.legend()

        fig_key = f"{ds}__{ker}"
        fig_name = f"part2_trace_{_sanitize_dataset_name(fig_key)}_{_short_hash(fig_key)}.png"
        fig_path = fig_dir / fig_name
        fig.savefig(fig_path, dpi=200)
        print("figure saved:", fig_path)

        plt.show()
        plt.close(fig)
else:
    print("Part2 plot skipped (no traces).")


In [ ]:
# ---- Optional Colab export / download / disconnect ----
# 本地运行：默认禁用（避免 Drive 相关操作）。
# 如需在 Colab 导出 bundle 到 Drive/下载，把 ENABLE_COLAB_EXPORT=True。
import json as _json
import shutil as _shutil
import time as _time
from datetime import timedelta

IS_COLAB = "google.colab" in sys.modules
ENABLE_COLAB_EXPORT = True

if not (IS_COLAB and ENABLE_COLAB_EXPORT):
    print("[note] export cell skipped (IS_COLAB=", IS_COLAB, ", ENABLE_COLAB_EXPORT=", ENABLE_COLAB_EXPORT, ")")
else:
    try:
        from google.colab import files, runtime
        _HAS_COLAB_EXPORT = True
    except Exception:
        files = None
        runtime = None
        _HAS_COLAB_EXPORT = False

    end_time = _time.time()
    try:
        elapsed_total = end_time - start_time
        time_str = str(timedelta(seconds=int(elapsed_total)))
    except NameError:
        time_str = "unknown (start_time was not set)"

    # 选择 zip 的命名基准：优先 Part2，其次 Part1 合并输出，其次 Part1 最后一个 run。
    out_dir_path = None
    if "part2_out_dir" in globals() and part2_out_dir is not None:
        out_dir_path = Path(part2_out_dir).resolve()
    elif "part1_merged_out_dir" in globals() and part1_merged_out_dir is not None:
        out_dir_path = Path(part1_merged_out_dir).resolve()
    elif "part1_run_meta" in globals() and part1_run_meta:
        try:
            out_dir_path = Path(part1_run_meta[-1]["out_dir"]).resolve()
        except Exception:
            out_dir_path = None

    if out_dir_path is None:
        out_dir_path = (BENCHMARK_DIR / "outputs").resolve()

    notebook_src_path = (Path(BENCHMARK_DIR) / "benchmark_tracecg_time_to_target_and_convergence.ipynb").resolve()
    if "DRIVE_OUTPUT_DIR" in globals():
        drive_output_dir = Path(DRIVE_OUTPUT_DIR)
    else:
        drive_output_dir = Path(DRIVE_MYDRIVE_DIR) / "EFGP_Eigenpro" / "benchmark_exports"
    drive_output_dir.mkdir(parents=True, exist_ok=True)

    zip_name = f"{out_dir_path.name}_bundle.zip"
    export_root = out_dir_path.parent / f"{out_dir_path.name}_export_bundle"
    zip_base = out_dir_path.parent / f"{out_dir_path.name}_bundle"
    zip_path = Path(f"{zip_base}.zip")
    target_drive_path = drive_output_dir / zip_name

    required_checks = {
        "out_dir_exists": out_dir_path.exists(),
        "notebook_exists": notebook_src_path.exists(),
        # 强制保证：如果跑了 Part2，就必须存在完整表（summary.csv / trace_all.csv）
        "part2_summary_exists": ("part2_out_dir" not in globals() or part2_out_dir is None) or (Path(part2_out_dir).resolve() / "summary.csv").exists(),
        "part2_trace_all_exists": ("part2_out_dir" not in globals() or part2_out_dir is None) or (Path(part2_out_dir).resolve() / "trace_all.csv").exists(),
        # 强制保证：如果跑了 Part1 合并表，就必须存在合并后的三张 CSV
        "part1_merged_ttt_exists": ("part1_merged_out_dir" not in globals() or part1_merged_out_dir is None) or (Path(part1_merged_out_dir).resolve() / "part1_time_to_target_merged.csv").exists(),
        "part1_merged_sb_exists": ("part1_merged_out_dir" not in globals() or part1_merged_out_dir is None) or (Path(part1_merged_out_dir).resolve() / "part1_same_time_budget_merged.csv").exists(),
        "part1_merged_hist_exists": ("part1_merged_out_dir" not in globals() or part1_merged_out_dir is None) or (Path(part1_merged_out_dir).resolve() / "part1_raw_eval_history_merged.csv").exists(),
    }
    print("artifact checks:", _json.dumps({k: bool(v) for k, v in required_checks.items()}, indent=2))
    if not bool(required_checks["out_dir_exists"]):
        raise FileNotFoundError(f"OUT_DIR not found: {out_dir_path}")
    # 若要求的关键表缺失，直接 fail，避免生成“看似成功但不完整”的 zip。
    missing = [k for k, v in required_checks.items() if not bool(v)]
    if missing:
        raise FileNotFoundError("Missing required artifacts: " + ", ".join(missing))

    if export_root.exists():
        _shutil.rmtree(export_root)
    export_root.mkdir(parents=True, exist_ok=True)

    includes: list[str] = []
    copied_src: set[str] = set()

    def _copytree_once(src: Path, dst: Path) -> None:
        src = Path(src).resolve()
        if not src.exists():
            return
        key = str(src)
        if key in copied_src:
            return
        copied_src.add(key)
        dst.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copytree(src, dst)
        includes.append(str(dst))

    # 1) 主输出目录（用于命名 zip 的那个 out_dir_path）
    export_primary_dir = export_root / "primary" / out_dir_path.name
    _copytree_once(out_dir_path, export_primary_dir)

    # 2) Part2 输出（如果存在）
    if "part2_out_dir" in globals() and part2_out_dir is not None:
        try:
            p2 = Path(part2_out_dir).resolve()
            _copytree_once(p2, export_root / "part2" / p2.name)
        except Exception as _e:
            print("[warn] failed to include part2_out_dir:", _e)

    # 3) Part1 合并表（如果存在）
    if "part1_merged_out_dir" in globals() and part1_merged_out_dir is not None:
        try:
            p1m = Path(part1_merged_out_dir).resolve()
            _copytree_once(p1m, export_root / "part1_merged" / p1m.name)
        except Exception as _e:
            print("[warn] failed to include part1_merged_out_dir:", _e)

    # 4) Part1 每个 target_delta 的原始输出目录（如果存在）
    if "part1_run_meta" in globals() and part1_run_meta:
        for ent in list(part1_run_meta):
            try:
                od = Path(ent.get("out_dir", "")).resolve()
                if od.exists():
                    _copytree_once(od, export_root / "part1_runs" / od.name)
            except Exception as _e:
                print("[warn] failed to include part1 run out_dir:", _e)

    # notebook 本体
    notebook_dst_dir = export_root / "notebook"
    notebook_dst_dir.mkdir(parents=True, exist_ok=True)
    notebook_dst_path = notebook_dst_dir / notebook_src_path.name
    if notebook_src_path.exists():
        _shutil.copy2(notebook_src_path, notebook_dst_path)

    export_manifest = {
        "run_tag": out_dir_path.name,
        "elapsed": time_str,
        "export_root": str(export_root),
        "out_dir": str(out_dir_path),
        "notebook_src": str(notebook_src_path),
        "drive_output_dir": str(drive_output_dir),
        "includes": includes + [str(notebook_dst_path) if notebook_src_path.exists() else "notebook_missing"],
    }
    (export_root / "export_manifest.json").write_text(_json.dumps(export_manifest, indent=2), encoding="utf-8")

    if zip_path.exists():
        zip_path.unlink()
    print(f"Packing export bundle: {export_root}")
    _shutil.make_archive(str(zip_base), "zip", root_dir=str(export_root.parent), base_dir=export_root.name)
    print("zip saved:", zip_path)

    print(f"Copying zip to Google Drive: {target_drive_path}")
    try:
        _shutil.copy2(zip_path, target_drive_path)
    except Exception as e:
        print("Drive copy failed:", e)

    local_ok = zip_path.exists() and zip_path.stat().st_size > 0
    print("local zip ok:", local_ok)

    if local_ok:
        if _HAS_COLAB_EXPORT and files is not None:
            print("Starting browser backup download...")
            files.download(str(zip_path))
        else:
            print("Not in Colab; browser download skipped.")

    print("-" * 30)
    print("Experiment status:", "bundle created" if local_ok else "bundle failed")
    print("Elapsed:", time_str)
    print("OUT_DIR:", out_dir_path)
    print("Local zip:", zip_path)
    print("Drive zip:", target_drive_path)
    print("-" * 30)

    if local_ok and _HAS_COLAB_EXPORT and runtime is not None:
        print("Disconnecting Colab runtime in 20 seconds to save GPU quota...")
        _time.sleep(20)
        runtime.unassign()
